> **対応するブログ記事**: [#5 OpenMS + AlphaPeptDeepで深層学習ベースのDIA解析を行う](../blog/article-05-openms.md)
>
> このNotebookはブログ記事 #5 のコードをセルごとに実行できるインタラクティブ版です。

# Step 5: OpenMS + AlphaPeptDeep DIA解析 → タンパク質マトリクス構築

In [ ]:
# 標準ライブラリ: ファイル操作(os)、パターン検索(glob)、正規表現(re)、外部コマンド実行(subprocess)、システム操作(sys)、時間計測(time)
import os, glob, re, subprocess, sys, time
import pandas as pd   # データフレーム操作ライブラリ（表形式データの読み書き・加工に使う）
import numpy as np    # 数値計算ライブラリ（配列演算・統計に使用）

In [ ]:
# --- パス設定（プロジェクト内の各ディレクトリ・ファイルへの相対パスを定数として定義） ---
MZML_DIR    = "../data/raw/raw_mzML"                # mzML生データが格納されているディレクトリ
FASTA_PATH  = "../data/raw/human_proteome.fasta"     # ヒトプロテオームのFASTA配列ファイルのパス
RESULTS_DIR = "../results"                           # 解析結果の出力先ルートディレクトリ

# OpenMS + AlphaPeptDeep パイプライン専用の出力ディレクトリ
OPENMS_OUT  = os.path.join(RESULTS_DIR, "openms_output")  # OpenMS関連の全出力をまとめるディレクトリ
LIBRARY_DIR = os.path.join(OPENMS_OUT, "library")          # AlphaPeptDeepが生成するスペクトルライブラリの保存先
OSWR_DIR    = os.path.join(OPENMS_OUT, "openswath")        # OpenSWATHの検索結果（.osw）の保存先
PYPROPHET_DIR = os.path.join(OPENMS_OUT, "pyprophet")      # PyProphetのスコアリング結果の保存先

# 各ディレクトリが存在しなければ作成する（exist_ok=Trueで既存でもエラーにならない）
for d in [OPENMS_OUT, LIBRARY_DIR, OSWR_DIR, PYPROPHET_DIR]:
    os.makedirs(d, exist_ok=True)

# mzMLファイルの一覧を取得してファイル名順にソート
mzml_files = sorted(glob.glob(os.path.join(MZML_DIR, "*.mzML")))
# 見つかったファイル数を表示（期待値: 32）
print(f"{len(mzml_files)} mzML files found")

In [ ]:
# --- パイプライン共通パラメータ ---
# ペプチド FDR（偽発見率）の閾値（論文と同じ 1% = 0.01）
Q_THRESHOLD = 0.01
# トリプシン消化のミスクリーベージ許容回数（論文と同じ: 最大1回の切断漏れを許容）
MISSED_CLEAVAGES = 1
# ペプチド長の下限と上限（論文のDIA-NNパラメータに合わせる）
PEPTIDE_MIN_LEN = 7
PEPTIDE_MAX_LEN = 45
# 前駆体イオンの電荷状態の範囲（2価〜4価のみ対象）
PRECURSOR_CHARGE_MIN = 2
PRECURSOR_CHARGE_MAX = 4
# フラグメントイオンのm/z範囲（質量分析計の測定範囲に合わせる）
FRAGMENT_MIN_MZ = 200.0
FRAGMENT_MAX_MZ = 1800.0
# 質量精度（ppm単位）: プリカーサーとフラグメントの質量誤差許容範囲
MS1_PPM = 10.0
MS2_PPM = 10.0

## Step 1: AlphaPeptDeepでスペクトルライブラリ生成

In [ ]:
# --- Step 1: AlphaPeptDeepによる予測スペクトルライブラリの生成 ---

# AlphaPeptDeepの設定をYAML形式で定義する
# peptdeep CLIは設定ファイル経由でパラメータを受け取る
PEPTDEEP_SETTINGS = os.path.join(OPENMS_OUT, "peptdeep_settings.yaml")

# 設定ファイルの内容を文字列として構築（YAMLフォーマット）
settings_content = f"""
# AlphaPeptDeep ライブラリ予測設定ファイル
model_mgr:
  # 使用するモデル: 汎用事前学習モデル（generic）
  external_ms2_model: ''
  external_rt_model: ''
  external_ccs_model: ''

library:
  # FASTAファイルのパス: ヒトプロテオーム配列
  fasta_files:
    - {os.path.abspath(FASTA_PATH)}
  # 出力ファイルのパス: OpenSWATH用のTSV形式
  output_tsv: {os.path.abspath(os.path.join(LIBRARY_DIR, "predicted_library.tsv"))}
  # 消化酵素の設定（トリプシン: K/Rの後で切断、Pの前は切断しない）
  enzyme: trypsin
  # ミスクリーベージの最大回数
  max_missed_cleavages: {MISSED_CLEAVAGES}
  # ペプチド長の範囲
  min_peptide_length: {PEPTIDE_MIN_LEN}
  max_peptide_length: {PEPTIDE_MAX_LEN}
  # 前駆体電荷の範囲
  min_precursor_charge: {PRECURSOR_CHARGE_MIN}
  max_precursor_charge: {PRECURSOR_CHARGE_MAX}
  # フラグメントイオンのm/z範囲
  min_fragment_mz: {FRAGMENT_MIN_MZ}
  max_fragment_mz: {FRAGMENT_MAX_MZ}
  # 固定修飾: システインのカルバミドメチル化（IAA処理による標準修飾）
  fix_modifications:
    - Carbamidomethyl@C
  # 可変修飾: なし（シンプルな設定で実行）
  var_modifications: []
  max_var_mod_num: 0
  # フラグメントイオンの種類: b/yイオン
  fragment_types:
    - b
    - y
  # フラグメントの最大電荷
  max_fragment_charge: 2
"""

# 設定ファイルをディスクに書き出す
with open(PEPTDEEP_SETTINGS, "w") as f:
    f.write(settings_content)
print(f"AlphaPeptDeep設定ファイルを保存: {PEPTDEEP_SETTINGS}")

In [ ]:
# peptdeep CLIコマンドを構築: FASTAからスペクトルライブラリを予測する
cmd_peptdeep = [
    "peptdeep",         # AlphaPeptDeepのCLIコマンド
    "library",          # ライブラリ予測モード
    "--settings",       # 設定ファイルを指定するフラグ
    PEPTDEEP_SETTINGS   # 設定ファイルのパス
]

print("Running AlphaPeptDeep library prediction...")
print(f"  FASTA: {FASTA_PATH}")
print(f"  Output: {LIBRARY_DIR}")
t0 = time.time()  # 実行開始時刻を記録（所要時間の計算用）

# peptdeep を子プロセスとして起動し、ログをリアルタイム出力する
# stdout=PIPEで標準出力をキャプチャ、stderr=STDOUTでエラー出力も統合、text=Trueで文字列として扱う
proc = subprocess.Popen(cmd_peptdeep, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
# peptdeepの出力を1行ずつ読み取り、Notebook上にリアルタイム表示するループ
for line in proc.stdout:
    sys.stdout.write(line)  # 各ログ行をそのまま標準出力に書き出す
proc.wait()  # プロセスの終了を待機（終了コードが確定する）

# 所要時間を分単位で表示し、終了コードも出力（0なら正常終了）
elapsed = (time.time() - t0) / 60
print(f"\nAlphaPeptDeep完了: {elapsed:.1f} min (exit code {proc.returncode})")

In [ ]:
# 生成されたスペクトルライブラリの確認
PREDICTED_LIB = os.path.join(LIBRARY_DIR, "predicted_library.tsv")
# ライブラリファイルが存在するか確認する
if os.path.exists(PREDICTED_LIB):
    # ライブラリをDataFrameとして読み込み（大きいファイルなのでnrows=5で先頭5行だけ確認）
    df_lib = pd.read_csv(PREDICTED_LIB, sep="\t", nrows=5)
    # ライブラリのファイルサイズをMB単位で表示する
    lib_size = os.path.getsize(PREDICTED_LIB) / 1024**2
    print(f"ライブラリ生成成功: {lib_size:.1f} MB")
    # ライブラリの列名を表示して、正しいフォーマットで生成されたか確認する
    print(f"列: {list(df_lib.columns)}")
    # 先頭5行を表示する
    df_lib.head()
else:
    # ファイルが存在しない場合はエラーメッセージを表示する
    print(f"ERROR: ライブラリファイルが見つかりません: {PREDICTED_LIB}")

## Step 2: OpenSWATHでDIA解析

In [ ]:
# --- Step 2: OpenSWATH による DIA 検索 ---

# ライブラリをOpenSWATH形式（.pqp）に変換する
# OpenSWATHはTSVライブラリも読めるが、PQP（SQLite）形式のほうが高速
PQP_LIB = os.path.join(LIBRARY_DIR, "predicted_library.pqp")

# OpenMS の TargetedFileConverter でTSV→PQP変換を行う
cmd_convert = [
    "TargetedFileConverter",  # OpenMSのファイル変換ツール
    "-in", PREDICTED_LIB,    # 入力: AlphaPeptDeepが生成したTSVライブラリ
    "-out", PQP_LIB          # 出力: OpenSWATH用のPQPライブラリ（SQLite形式）
]

print("TSVライブラリをPQP形式に変換中...")
t0 = time.time()  # 変換開始時刻を記録

# TargetedFileConverterを子プロセスとして実行する
proc = subprocess.Popen(cmd_convert, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
# 出力を1行ずつ表示する
for line in proc.stdout:
    sys.stdout.write(line)
proc.wait()  # プロセスの終了を待機する

# 変換の所要時間と終了コードを表示する
print(f"PQP変換完了: {(time.time() - t0):.1f}s (exit code {proc.returncode})")

In [ ]:
# 各mzMLファイルに対してOpenSWATHを実行する関数
def run_openswath(mzml_path, pqp_lib, output_dir, ms1_ppm, ms2_ppm):
    """1つのmzMLファイルに対してOpenSwathWorkflowを実行する。

    Args:
        mzml_path: 入力mzMLファイルのパス
        pqp_lib: PQP形式のスペクトルライブラリのパス
        output_dir: 出力ディレクトリ（.oswファイルの保存先）
        ms1_ppm: MS1の質量精度（ppm）
        ms2_ppm: MS2の質量精度（ppm）

    Returns:
        出力.oswファイルのパス
    """
    # 入力ファイル名から拡張子を除去してサンプル名を取得する（例: CRC01-N）
    sample_name = os.path.splitext(os.path.basename(mzml_path))[0]
    # 出力.oswファイルのパスを構築する（各サンプルごとに個別のファイル）
    osw_out = os.path.join(output_dir, f"{sample_name}.osw")

    # OpenSwathWorkflowのコマンドライン引数を構築する
    cmd = [
        "OpenSwathWorkflow",          # OpenSWATHのメインコマンド
        "-in", mzml_path,             # 入力: DIAデータのmzMLファイル
        "-tr", pqp_lib,               # ライブラリ: PQP形式の予測スペクトルライブラリ
        "-out_osw", osw_out,          # 出力: OSW形式（SQLite）の結果ファイル
        "-min_upper_edge_dist", "1",  # DIA窓境界からの最小距離（品質フィルタ）
        "-mz_extraction_window", str(ms2_ppm),       # MS2のm/z抽出窓（ppm単位）
        "-mz_extraction_window_unit", "ppm",          # m/z抽出窓の単位をppmに指定
        "-mz_extraction_window_ms1", str(ms1_ppm),   # MS1のm/z抽出窓（ppm単位）
        "-mz_extraction_window_ms1_unit", "ppm",      # MS1のm/z抽出窓の単位をppmに指定
        "-use_ms1_traces",            # MS1レベルのXIC（抽出イオンクロマトグラム）も使用する
        "-Scoring:stop_report_after_feature", "5",    # 各遷移グループで報告する特徴量の最大数
        "-Scoring:TransitionGroupPicker:min_peak_width", "10",  # ピークの最小幅（秒）
        "-threads", "4"               # 並列処理のスレッド数（環境に応じて調整可能）
    ]

    # OpenSwathWorkflowを子プロセスとして実行する
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    # 出力を1行ずつリアルタイム表示する
    for line in proc.stdout:
        sys.stdout.write(line)
    proc.wait()  # プロセスの終了を待機する

    return osw_out  # 出力ファイルのパスを返す

In [ ]:
# 全32サンプルに対してOpenSWATHを順次実行する
print(f"OpenSWATH実行開始: {len(mzml_files)} ファイル")
t0_all = time.time()  # 全体の開始時刻を記録する

# 各サンプルの出力ファイルパスを格納するリスト
osw_files = []

# 全mzMLファイルに対してOpenSWATHを順番に実行するループ
for i, mzml_path in enumerate(mzml_files):
    # 現在の処理ファイル番号とファイル名を表示する
    sample_name = os.path.basename(mzml_path)
    print(f"\n[{i+1}/{len(mzml_files)}] Processing: {sample_name}")
    t0 = time.time()  # 個別ファイルの開始時刻を記録する

    # OpenSWATHを実行し、出力ファイルパスを取得する
    osw_path = run_openswath(mzml_path, PQP_LIB, OSWR_DIR, MS1_PPM, MS2_PPM)
    # 出力ファイルパスをリストに追加する
    osw_files.append(osw_path)

    # 個別ファイルの処理時間を分単位で表示する
    elapsed = (time.time() - t0) / 60
    print(f"  完了: {elapsed:.1f} min")

# 全体の所要時間を表示する
total_elapsed = (time.time() - t0_all) / 60
print(f"\nOpenSWATH全完了: {total_elapsed:.1f} min ({len(osw_files)} ファイル)")

## Step 3: PyProphetでFDR推定

In [ ]:
# --- Step 3: PyProphet によるスコアリングとFDR推定 ---

# 全サンプルの.oswファイルを1つに統合する（PyProphetはマルチファイル入力に対応）
# 統合することで、サンプル間の情報を利用したglobal FDR推定が可能になる
MERGED_OSW = os.path.join(PYPROPHET_DIR, "merged.osw")

# pyprophet merge コマンドで全.oswファイルを1つに統合する
cmd_merge = [
    "pyprophet", "merge",       # PyProphetのマージモード
    "--out", MERGED_OSW,        # 統合後のファイルパス
] + osw_files                   # 入力: 全32サンプルの.oswファイル

print("PyProphet: .oswファイルを統合中...")
t0 = time.time()  # 統合開始時刻を記録する

# pyprophet merge を子プロセスとして実行する
proc = subprocess.Popen(cmd_merge, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
# 出力を1行ずつ表示する
for line in proc.stdout:
    sys.stdout.write(line)
proc.wait()  # 統合の完了を待機する

print(f"統合完了: {(time.time() - t0):.1f}s (exit code {proc.returncode})")

In [ ]:
# MS2レベルのスコアリング: 各ピークグループにd-scoreとq-valueを付与する
cmd_score = [
    "pyprophet", "score",        # PyProphetのスコアリングモード
    "--in", MERGED_OSW,          # 入力: 統合された.oswファイル
    "--level", "ms2",            # スコアリングレベル: MS2（フラグメントイオンレベル）
    "--ss_initial_fdr", "0.15",  # semi-supervised learningの初期FDR閾値
    "--ss_iteration_fdr", "0.05" # 繰り返し学習時のFDR閾値
]

print("PyProphet: MS2レベルスコアリング中...")
t0 = time.time()  # スコアリング開始時刻を記録する

# pyprophet score を子プロセスとして実行する
proc = subprocess.Popen(cmd_score, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
# 出力を1行ずつ表示する
for line in proc.stdout:
    sys.stdout.write(line)
proc.wait()  # スコアリングの完了を待機する

print(f"MS2スコアリング完了: {(time.time() - t0) / 60:.1f} min (exit code {proc.returncode})")

In [ ]:
# ペプチドレベルのFDR推定: MS2スコアをペプチドレベルに統合する
cmd_peptide = [
    "pyprophet", "peptide",      # ペプチドレベルのFDR推定モード
    "--in", MERGED_OSW,          # 入力: スコアリング済みの.oswファイル
    "--context", "global"        # FDRのコンテキスト: 全サンプルを通じた global FDR
]

print("PyProphet: ペプチドレベルFDR推定中...")
t0 = time.time()  # 開始時刻を記録する

# pyprophet peptide を子プロセスとして実行する
proc = subprocess.Popen(cmd_peptide, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    sys.stdout.write(line)
proc.wait()  # 完了を待機する

print(f"ペプチドFDR完了: {(time.time() - t0):.1f}s (exit code {proc.returncode})")

In [ ]:
# タンパク質レベルのFDR推定: ペプチドスコアをタンパク質レベルに統合する
cmd_protein = [
    "pyprophet", "protein",      # タンパク質レベルのFDR推定モード
    "--in", MERGED_OSW,          # 入力: ペプチドFDR推定済みの.oswファイル
    "--context", "global"        # FDRのコンテキスト: global FDR
]

print("PyProphet: タンパク質レベルFDR推定中...")
t0 = time.time()  # 開始時刻を記録する

# pyprophet protein を子プロセスとして実行する
proc = subprocess.Popen(cmd_protein, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    sys.stdout.write(line)
proc.wait()  # 完了を待機する

print(f"タンパク質FDR完了: {(time.time() - t0):.1f}s (exit code {proc.returncode})")

In [ ]:
# スコアリング結果をTSV形式でエクスポートする（後段のPython処理で読み込むため）
EXPORT_TSV = os.path.join(PYPROPHET_DIR, "pyprophet_export.tsv")

cmd_export = [
    "pyprophet", "export",         # エクスポートモード
    "--in", MERGED_OSW,            # 入力: 全レベルのFDR推定が完了した.oswファイル
    "--out", EXPORT_TSV,           # 出力: TSV形式のエクスポートファイル
    "--max_global_peptide_qvalue", str(Q_THRESHOLD),   # ペプチドFDRフィルタ（1%以下）
    "--max_global_protein_qvalue", str(Q_THRESHOLD)    # タンパク質FDRフィルタ（1%以下）
]

print("PyProphet: 結果をTSVにエクスポート中...")
t0 = time.time()  # 開始時刻を記録する

# pyprophet export を子プロセスとして実行する
proc = subprocess.Popen(cmd_export, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    sys.stdout.write(line)
proc.wait()  # 完了を待機する

# エクスポートファイルのサイズと所要時間を表示する
if os.path.exists(EXPORT_TSV):
    export_size = os.path.getsize(EXPORT_TSV) / 1024**2
    print(f"エクスポート完了: {export_size:.1f} MB, {(time.time() - t0):.1f}s")
else:
    print("ERROR: エクスポートファイルが生成されませんでした")

## Step 4: タンパク質マトリクス構築

In [ ]:
# FASTAからGene Symbolマッピング辞書を構築する（sage記事と同じ関数）
def parse_fasta_gene_map(fasta_path):
    """UniProt FASTA ヘッダから accession -> Gene Symbol の辞書を作る。

    ヘッダ例: >sp|P04637|P53_HUMAN ... GN=TP53 ...
    GN= がなければ Entry Name の先頭語 (例: P53) を使う。
    """
    id_to_gene = {}  # アクセッション番号→Gene Symbolの辞書を初期化する
    # 「GN=」の後に続く空白以外の文字列（=Gene Symbol）を抽出する正規表現パターン
    gn_re = re.compile(r"\bGN=(\S+)")
    # FASTAファイルを開いて1行ずつ読み込む
    with open(fasta_path) as f:
        for line in f:  # FASTAファイルの全行を走査する
            if not line.startswith(">"):
                continue  # アミノ酸配列行はスキップ（ヘッダ行のみ処理する）
            # ヘッダ行を「|」で最大3つに分割する（例: "sp", "P04637", "P53_HUMAN ..."）
            parts = line[1:].split("|", 2)
            if len(parts) >= 3:
                acc = parts[1]                     # アクセッション番号（例: P04637）
                entry_name = parts[2].split()[0]   # エントリ名（例: P53_HUMAN）
            else:
                acc = line[1:].split()[0]           # 非標準ヘッダの場合のフォールバック
                entry_name = acc
            m = gn_re.search(line)  # 「GN=遺伝子名」パターンを検索する
            # GN=が見つかればその遺伝子名、なければエントリ名の「_」前を使用する
            id_to_gene[acc] = m.group(1) if m else entry_name.split("_")[0]
    return id_to_gene  # {アクセッション番号: Gene Symbol} の辞書を返す

# FASTAを解析してGene Symbolマッピングを構築する
id_to_gene = parse_fasta_gene_map(FASTA_PATH)
print(f"FASTA entries: {len(id_to_gene)}")

In [ ]:
# --- Step 4: タンパク質マトリクスの構築 ---

# PyProphetのエクスポートTSVを読み込む
df = pd.read_csv(EXPORT_TSV, sep="\t")
print(f"PyProphet export: {len(df)} rows, {len(df.columns)} columns")
# 列名を表示して、必要なカラムが含まれているか確認する
print(f"列名: {list(df.columns[:10])} ...")

# PyProphetのエクスポートには以下の主要列が含まれる:
#   - ProteinName: タンパク質名（UniProtアクセッション）
#   - PeptideSequence: ペプチド配列（修飾含む）
#   - filename: 元のmzMLファイル名
#   - Intensity: ペプチドの定量強度
#   - m_score: PyProphetのq-value（FDR）

# filenameカラムからサンプル名を抽出する（ディレクトリパスと拡張子を除去）
df["sample"] = df["filename"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])
# サンプル名の一覧を表示して正しく抽出できているか確認する
print(f"サンプル数: {df['sample'].nunique()}")
print(f"サンプル一覧: {sorted(df['sample'].unique())[:5]} ...")

# ProteinNameからアクセッション番号を抽出し、Gene Symbolに変換する
# PyProphetの出力では ProteinName が "1/sp|P04637|P53_HUMAN" のような形式になることがある
df["acc"] = df["ProteinName"].str.extract(r"sp\|(\w+)\|", expand=False)
# 抽出できなかった場合はProteinNameをそのまま使う
df["acc"] = df["acc"].fillna(df["ProteinName"])
# アクセッション番号をGene Symbolに変換する（マッピングできない場合はアクセッション番号を維持）
df["gene"] = df["acc"].map(id_to_gene).fillna(df["acc"])

# 変換結果を確認する
print(f"Gene Symbol変換: {df['gene'].nunique()} ユニーク遺伝子")

# ペプチドレベルの強度をタンパク質レベルに集約する
# 各サンプル×各タンパク質について、ペプチド強度の合計を計算する

# ピボットテーブルを構築: 行=gene（タンパク質）、列=sample、値=Intensityの合計
matrix = df.pivot_table(
    index="gene",          # 行: Gene Symbol（タンパク質名）
    columns="sample",      # 列: サンプル名（CRC01-N, CRC01-Tなど）
    values="Intensity",    # 値: ペプチドの定量強度
    aggfunc="sum"          # 集約方法: 同一タンパク質の全ペプチド強度を合計
)

# 値が0の箇所をNaN（未検出）に置換する
matrix = matrix.replace(0, np.nan)
# 全サンプルでNaN（全く検出されなかった）のタンパク質行を削除する
matrix = matrix.dropna(how="all")
# インデックス名を設定する（CSV出力時のヘッダになる）
matrix.index.name = "Protein"

# マトリクスのサイズを表示する（行数=タンパク質数、列数=サンプル数）
print(f"タンパク質マトリクス: {matrix.shape[0]} proteins x {matrix.shape[1]} samples")
# 先頭5行を表示して中身を確認する
matrix.head()

In [ ]:
# タンパク質マトリクスをCSVファイルとして保存する
out_csv = os.path.join(RESULTS_DIR, "protein_matrix_from_openms.csv")
# DataFrameをCSV形式で保存する（インデックス=タンパク質名も含める）
matrix.to_csv(out_csv)
# 保存先パスとマトリクスサイズを表示して確認する
print(f"保存完了: {out_csv}")
print(f"  {matrix.shape[0]} タンパク質 × {matrix.shape[1]} サンプル")

## 完了

OpenMS + AlphaPeptDeep パイプラインによるDIA解析が完了しました。

出力ファイル `protein_matrix_from_openms.csv` は sage の出力と同じ形式（タンパク質 × サンプル）なので、
#6 の前処理ステップにそのまま接続できます。